In [1]:
#Required Linbraries
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
!pip install requests beautifulsoup4 pandas lxml selenium webdriver-manager

In [12]:
URL = "https://www.goodreads.com/list/show/1.Best_Books_Ever?page=1"

options = webdriver.ChromeOptions()
options.add_argument("--disable-blink-features=AutomationEnabled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

driver.get(URL)
time.sleep(15)

soup = BeautifulSoup(driver.page_source, "lxml")
driver.quit()

table = soup.find("table", {"class": "tableList"})
print("Found table:", table is not None)

if table:
    rows = table.find_all("tr")
    print("Rows found:", len(rows))

    first_row = rows[0]
    title  = first_row.find("a", class_="bookTitle")
    author = first_row.find("a", class_="authorName")
    rating = first_row.find("span", class_="minirating")

    print("Title:",  title.get_text(strip=True)  if title  else "NOT FOUND")
    print("Author:", author.get_text(strip=True) if author else "NOT FOUND")
    print("Rating:", rating.get_text(strip=True) if rating else "NOT FOUND")


Found table: True
Rows found: 100
Title: The Hunger Games (The Hunger Games, #1)
Author: Suzanne Collins
Rating: 4.35 avg rating — 10,115,839 ratings


In [ ]:
#Book scraping function
def scrape_goodreads(total_books=1000):
    all_books = []
    total_pages = total_books // 100  # 10 pages

    for page_num in range(1, total_pages + 1):
        print(f"Scraping page {page_num}/{total_pages}...")

        url = f"https://www.goodreads.com/list/show/1.Best_Books_Ever?page={page_num}"

        options = webdriver.ChromeOptions()
        options.add_argument("--disable-blink-features=AutomationEnabled")
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option("useAutomationExtension", False)

        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
        driver.get(url)
        time.sleep(15)  # wait for JS to load

        soup = BeautifulSoup(driver.page_source, "lxml")
        driver.quit()

        table = soup.find("table", {"class": "tableList"})
        if not table:
            print(f"  ✗ No table found on page {page_num}, skipping.")
            continue

        rows = table.find_all("tr")
        for row in rows:
            title_tag  = row.find("a", class_="bookTitle")
            author_tag = row.find("a", class_="authorName")
            rating_tag = row.find("span", class_="minirating")
            rank_tag   = row.find("td", class_="number")
            img_tag    = row.find("img")

            # avg rating and num ratings are both inside minirating text
            avg_rating, num_ratings = None, None
            if rating_tag:
                rating_text = rating_tag.get_text(strip=True)
                try:
                    avg_rating = float(rating_text.split()[0])
                except:
                    pass
                try:
                    num_ratings = int(rating_text.split("—")[1].strip().split()[0].replace(",", ""))
                except:
                    pass

            book = {
                "rank":           rank_tag.get_text(strip=True)       if rank_tag   else None,
                "title":          title_tag.get_text(strip=True)      if title_tag  else None,
                "author":         author_tag.get_text(strip=True)     if author_tag else None,
                "avg_rating":     avg_rating,
                "num_ratings":    num_ratings,
                "cover_img_url":  img_tag["src"]                      if img_tag    else None,
                "goodreads_url":  "https://www.goodreads.com" + title_tag["href"] if title_tag else None,
            }

            if book["title"]:
                all_books.append(book)

        print(f"  ✓ Page {page_num} done — total so far: {len(all_books)}")
        time.sleep(random.uniform(3, 6))  # polite delay between pages

    return pd.DataFrame(all_books)


df = scrape_goodreads(1000)
print(f"\nDone! {len(df)} books collected.")
print(df.head())

Scraping page 1/10...
  ✓ Page 1 done — total so far: 100
Scraping page 2/10...
  ✓ Page 2 done — total so far: 200
Scraping page 3/10...
  ✓ Page 3 done — total so far: 300
Scraping page 4/10...
  ✓ Page 4 done — total so far: 400
Scraping page 5/10...
  ✓ Page 5 done — total so far: 500
Scraping page 6/10...
  ✓ Page 6 done — total so far: 600
Scraping page 7/10...
  ✓ Page 7 done — total so far: 700
Scraping page 8/10...
  ✓ Page 8 done — total so far: 800
Scraping page 9/10...
  ✓ Page 9 done — total so far: 900
Scraping page 10/10...
  ✓ Page 10 done — total so far: 1000

Done! 1000 books collected.
  rank                                              title           author  \
0    1            The Hunger Games (The Hunger Games, #1)  Suzanne Collins   
1    2                                Pride and Prejudice      Jane Austen   
2    3                              To Kill a Mockingbird       Harper Lee   
3    4  Harry Potter and the Order of the Phoenix (Har...     J.K. Rowling  

In [14]:
df["source"] = "GoodReads"
df.to_csv("goodreads_1000.csv", index=False)
print("Saved to goodreads_1000.csv")

Saved to goodreads_1000.csv


In [ ]:
#adding features
# Load existing CSV
df = pd.read_csv("goodreads_1000.csv")

# Test on just the first book
test_url = df["goodreads_url"][0]
print("Testing on:", test_url)

options = webdriver.ChromeOptions()
options.add_argument("--disable-blink-features=AutomationEnabled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
driver.get(test_url)
time.sleep(10)

soup = BeautifulSoup(driver.page_source, "lxml")
driver.quit()

# Year published
year_tag = soup.find("p", {"data-testid": "publicationInfo"})
print("Year:", year_tag.get_text(strip=True) if year_tag else "NOT FOUND")

# Genres
genre_tags = soup.find_all("span", {"data-testid": "genre"})
genres = [g.get_text(strip=True) for g in genre_tags]
print("Genres:", genres if genres else "NOT FOUND")

# Description
desc_tag = soup.find("span", {"data-testid": "description"})
print("Description:", desc_tag.get_text(strip=True)[:200] if desc_tag else "NOT FOUND")

# Number of pages
pages_tag = soup.find("p", {"data-testid": "pagesFormat"})
print("Pages:", pages_tag.get_text(strip=True) if pages_tag else "NOT FOUND")

# Series
series_tag = soup.find("h3", {"aria-label": lambda x: x and "series" in x.lower()})
print("Series:", series_tag.get_text(strip=True) if series_tag else "NOT FOUND")

Testing on: https://www.goodreads.com/book/show/2767052-the-hunger-games
Year: First published September 14, 2008
Genres: NOT FOUND
Description: NOT FOUND
Pages: 374 pages, Hardcover
Series: The Hunger Games#1


In [ ]:
# hunt for the correct tags for genre and description
options = webdriver.ChromeOptions()
options.add_argument("--disable-blink-features=AutomationEnabled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
driver.get("https://www.goodreads.com/book/show/2767052-the-hunger-games")
time.sleep(15)  # extra time for JS to load

soup = BeautifulSoup(driver.page_source, "lxml")
driver.quit()

# Search for description — trying alternative tags
print("=== DESCRIPTION ATTEMPTS ===")
for tag in ["description", "synopsis", "blurb"]:
    result = soup.find("span", {"data-testid": tag})
    print(f"data-testid='{tag}':", result.get_text(strip=True)[:100] if result else "NOT FOUND")

# finding any div/span that contains the word "Katniss" (it's in the description)
katniss = soup.find(string=lambda text: text and "Katniss" in text)
print("\nFound 'Katniss' in page:", katniss[:200] if katniss else "NOT FOUND")

# Search for genres
print("\n=== GENRE ATTEMPTS ===")
for class_name in ["BookPageMetadataSection__genres", "genres", "genresList"]:
    result = soup.find(class_=class_name)
    print(f"class='{class_name}':", result.get_text(strip=True)[:100] if result else "NOT FOUND")

=== DESCRIPTION ATTEMPTS ===
data-testid='description': NOT FOUND
data-testid='synopsis': NOT FOUND
data-testid='blurb': NOT FOUND

Found 'Katniss' in page: Sixteen-year-old Katniss Everdeen regards it as a death sentence when she steps forward to take her sister's place in the Games. But Katniss has been close to dead before-and survival, for her, is sec

=== GENRE ATTEMPTS ===
class='BookPageMetadataSection__genres': GenresYoung AdultDystopiaFictionFantasyScience FictionRomanceAdventure...more
class='genres': NOT FOUND
class='genresList': NOT FOUND


In [17]:
# Description — find the parent of the Katniss text
katniss_text = soup.find(string=lambda text: text and "Katniss" in text)
desc = katniss_text.find_parent().get_text(strip=True) if katniss_text else None
print("Description:", desc[:200] if desc else "NOT FOUND")

# Genres — find all <a> tags inside the genres div
genres_div = soup.find(class_="BookPageMetadataSection__genres")
genre_links = genres_div.find_all("a") if genres_div else []
genres = [g.get_text(strip=True) for g in genre_links]
print("Genres:", genres)

Description: Winning means fame and fortune. Losing means certain death. The Hunger Games have begun. . . .In the ruins of a place once known as North America lies the nation of Panem, a shining Capitol surrounded
Genres: ['Young Adult', 'Dystopia', 'Fiction', 'Fantasy', 'Science Fiction', 'Romance', 'Adventure']


In [24]:
options = webdriver.ChromeOptions()
options.add_argument("--disable-blink-features=AutomationEnabled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Test on 1 book
details = scrape_book_details(df["goodreads_url"][0], driver)
print(details)

{'year_published': 'First published September 14, 2008', 'genres': 'Young Adult, Dystopia, Fiction, Fantasy, Science Fiction, Romance, Adventure', 'description': ':root{--toastify-color-light: #fff;--toastify-color-dark: #121212;--toastify-color-info: #3498db;--toastify-color-success: #07bc0c;--toastify-color-warning: #f1c40f;--toastify-color-error: hsl(6, 78%, 57%);--toastify-color-transparent: rgba(255, 255, 255, .7);--toastify-icon-color-info: var(--toastify-color-info);--toastify-icon-color-success: var(--toastify-color-success);--toastify-icon-color-warning: var(--toastify-color-warning);--toastify-icon-color-error: var(--toastify-color-error);--toastify-container-width: fit-content;--toastify-toast-width: 320px;--toastify-toast-offset: 16px;--toastify-toast-top: max(var(--toastify-toast-offset), env(safe-area-inset-top));--toastify-toast-right: max(var(--toastify-toast-offset), env(safe-area-inset-right));--toastify-toast-left: max(var(--toastify-toast-offset), env(safe-area-inse

In [25]:
def scrape_book_details(url, driver):
    driver.get(url)
    time.sleep(random.uniform(8, 12))
    
    soup = BeautifulSoup(driver.page_source, "lxml")
    
    # Year published
    year_tag = soup.find("p", {"data-testid": "publicationInfo"})
    year = year_tag.get_text(strip=True) if year_tag else None

    # Genres
    genres_div = soup.find(class_="BookPageMetadataSection__genres")
    genre_links = genres_div.find_all("a") if genres_div else []
    genres = ", ".join([g.get_text(strip=True) for g in genre_links])

    # Description — fixed: look for the specific container div
    desc = None
    desc_div = soup.find("div", {"data-testid": "description"})
    if desc_div:
        desc = desc_div.get_text(strip=True)
    else:
        # fallback: find text containing known book-description words, excluding CSS
        for tag in soup.find_all(["span", "div"]):
            text = tag.get_text(strip=True)
            if len(text) > 100 and len(text) < 2000 and "{" not in text:
                desc = text
                break

    # Pages
    pages_tag = soup.find("p", {"data-testid": "pagesFormat"})
    pages = pages_tag.get_text(strip=True) if pages_tag else None

    # Series
    series_tag = soup.find("h3", {"aria-label": lambda x: x and "series" in x.lower()})
    series = series_tag.get_text(strip=True) if series_tag else None

    return {
        "year_published": year,
        "genres":         genres,
        "description":    desc,
        "pages":          pages,
        "series":         series
    }

# Test again on 1 book
details = scrape_book_details(df["goodreads_url"][0], driver)
print("Year:", details["year_published"])
print("Genres:", details["genres"])
print("Description:", details["description"][:200] if details["description"] else "NOT FOUND")
print("Pages:", details["pages"])
print("Series:", details["series"])

Year: First published September 14, 2008
Genres: Young Adult, Dystopia, Fiction, Fantasy, Science Fiction, Romance, Adventure
Description: Winning means fame and fortune. Losing means certain death. The Hunger Games have begun. . . .In the ruins of a place once known as North America lies the nation of Panem, a shining Capitol surrounded
Pages: 374 pages, Hardcover
Series: The Hunger Games#1


In [26]:
for i, row in df.iterrows():
    print(f"[{i+1}/1000] {row['title']}")
    try:
        details = scrape_book_details(row["goodreads_url"], driver)
        for field, value in details.items():
            df.at[i, field] = value
    except Exception as e:
        print(f"  ✗ Error: {e}")

    # Save every 50 books
    if (i + 1) % 50 == 0:
        df.to_csv("goodreads_1000_full.csv", index=False)
        print(f"  ✓ Progress saved at {i+1} books")

# Final save
df["source"] = "GoodReads"
df.to_csv("goodreads_1000_full.csv", index=False)
print("Done! Saved to goodreads_1000_full.csv")

[1/1000] The Hunger Games (The Hunger Games, #1)
[2/1000] Pride and Prejudice
[3/1000] To Kill a Mockingbird
[4/1000] Harry Potter and the Order of the Phoenix (Harry Potter, #5)
[5/1000] The Book Thief
[6/1000] Twilight (Twilight Saga, #1)
[7/1000] Animal Farm
[8/1000] J.R.R. Tolkien 4-Book Boxed Set: The Hobbit and The Lord of the Rings
[9/1000] The Chronicles of Narnia (The Chronicles of Narnia, #1-7)
[10/1000] The Fault in Our Stars
[11/1000] The Picture of Dorian Gray
[12/1000] The Lightning Thief (Percy Jackson and the Olympians, #1)
[13/1000] Wuthering Heights
[14/1000] The Giving Tree
[15/1000] Harry Potter and the Order of the Phoenix (Harry Potter, #5)
[16/1000] The Perks of Being a Wallflower
[17/1000] The Little Prince
[18/1000] Gone with the Wind
[19/1000] Jane Eyre
[20/1000] The Great Gatsby
[21/1000] Crime and Punishment
[22/1000] The Da Vinci Code (Robert Langdon, #2)
[23/1000] Alice’s Adventures in Wonderland / Through the Looking-Glass
[24/1000] Divergent (Divergent, 

In [27]:
# Check what columns got filled in for book 1 (a completed book)
print(df.iloc[0])

rank                                                              1
title                       The Hunger Games (The Hunger Games, #1)
author                                              Suzanne Collins
avg_rating                                                     4.35
num_ratings                                                10115839
cover_img_url     https://i.gr-assets.com/images/S/compressed.ph...
goodreads_url     https://www.goodreads.com/book/show/2767052-th...
source                                                    GoodReads
year_published                   First published September 14, 2008
genres            Young Adult, Dystopia, Fiction, Fantasy, Scien...
description       Winning means fame and fortune. Losing means c...
pages                                          374 pages, Hardcover
series                                           The Hunger Games#1
Name: 0, dtype: object


In [29]:
for i, row in df.iterrows():
    if pd.notna(row["author"]) and row["author"] != "":
        print(f"[{i+1}/1000] ✓ Already scraped: {row['title']}")
        continue

    print(f"[{i+1}/1000] Scraping: {row['title']}")
    try:
        details = scrape_book_details(row["goodreads_url"], driver)
        for field, value in details.items():
            df.at[i, field] = value
    except Exception as e:
        print(f"  ✗ Error: {e}")
        try:
            driver.quit()
        except:
            pass
        driver = create_driver()

[1/1000] ✓ Already scraped: The Hunger Games (The Hunger Games, #1)
[2/1000] ✓ Already scraped: Pride and Prejudice
[3/1000] ✓ Already scraped: To Kill a Mockingbird
[4/1000] ✓ Already scraped: Harry Potter and the Order of the Phoenix (Harry Potter, #5)
[5/1000] ✓ Already scraped: The Book Thief
[6/1000] ✓ Already scraped: Twilight (Twilight Saga, #1)
[7/1000] ✓ Already scraped: Animal Farm
[8/1000] ✓ Already scraped: J.R.R. Tolkien 4-Book Boxed Set: The Hobbit and The Lord of the Rings
[9/1000] ✓ Already scraped: The Chronicles of Narnia (The Chronicles of Narnia, #1-7)
[10/1000] ✓ Already scraped: The Fault in Our Stars
[11/1000] ✓ Already scraped: The Picture of Dorian Gray
[12/1000] ✓ Already scraped: The Lightning Thief (Percy Jackson and the Olympians, #1)
[13/1000] ✓ Already scraped: Wuthering Heights
[14/1000] ✓ Already scraped: The Giving Tree
[15/1000] ✓ Already scraped: Harry Potter and the Order of the Phoenix (Harry Potter, #5)
[16/1000] ✓ Already scraped: The Perks of Be

In [31]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import pandas as pd

def create_driver():
    options = Options()
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    return webdriver.Chrome(options=options)

df = pd.read_csv("goodreads_1000_full.csv")
driver = create_driver()

for i, row in df.iterrows():
    if pd.notna(row["author"]) and row["author"] != "":
        print(f"[{i+1}/1000] ✓ Already scraped: {row['title']}")
        continue

    print(f"[{i+1}/1000] Scraping: {row['title']}")
    try:
        details = scrape_book_details(row["goodreads_url"], driver)
        for field, value in details.items():
            df.at[i, field] = value
    except Exception as e:
        print(f"  ✗ Error: {e}")
        try:
            driver.quit()
        except:
            pass
        driver = create_driver()

    if (i + 1) % 50 == 0:
        df.to_csv("goodreads_1000_full.csv", index=False)
        print(f"  ✓ Progress saved at {i+1} books")

driver.quit()
df["source"] = "GoodReads"
df.to_csv("goodreads_1000_full.csv", index=False)
print("Done! Saved to goodreads_1000_full.csv")

[1/1000] ✓ Already scraped: The Hunger Games (The Hunger Games, #1)
[2/1000] ✓ Already scraped: Pride and Prejudice
[3/1000] ✓ Already scraped: To Kill a Mockingbird
[4/1000] ✓ Already scraped: Harry Potter and the Order of the Phoenix (Harry Potter, #5)
[5/1000] ✓ Already scraped: The Book Thief
[6/1000] ✓ Already scraped: Twilight (Twilight Saga, #1)
[7/1000] ✓ Already scraped: Animal Farm
[8/1000] ✓ Already scraped: J.R.R. Tolkien 4-Book Boxed Set: The Hobbit and The Lord of the Rings
[9/1000] ✓ Already scraped: The Chronicles of Narnia (The Chronicles of Narnia, #1-7)
[10/1000] ✓ Already scraped: The Fault in Our Stars
[11/1000] ✓ Already scraped: The Picture of Dorian Gray
[12/1000] ✓ Already scraped: The Lightning Thief (Percy Jackson and the Olympians, #1)
[13/1000] ✓ Already scraped: Wuthering Heights
[14/1000] ✓ Already scraped: The Giving Tree
[15/1000] ✓ Already scraped: Harry Potter and the Order of the Phoenix (Harry Potter, #5)
[16/1000] ✓ Already scraped: The Perks of Be

In [32]:
print(df.iloc[114])  # row 115
print(df.iloc[113])  # row 114

rank                                                            115
title                              Perfume: The Story of a Murderer
author                                              Patrick Süskind
avg_rating                                                     4.04
num_ratings                                                  571785
cover_img_url     https://i.gr-assets.com/images/S/compressed.ph...
goodreads_url       https://www.goodreads.com/book/show/343.Perfume
source                                                    GoodReads
year_published                                                  NaN
genres                                                          NaN
description                                                     NaN
pages                                                           NaN
series                                                          NaN
Name: 114, dtype: object
rank                                                            114
title                  

In [33]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import pandas as pd

def create_driver():
    options = Options()
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    return webdriver.Chrome(options=options)

df = pd.read_csv("goodreads_1000_full.csv")
driver = create_driver()

for i, row in df.iterrows():
    if i < 114:
        print(f"[{i+1}/1000] ✓ Already scraped: {row['title']}")
        continue

    print(f"[{i+1}/1000] Scraping: {row['title']}")
    try:
        details = scrape_book_details(row["goodreads_url"], driver)
        for field, value in details.items():
            df.at[i, field] = value
    except Exception as e:
        print(f"  ✗ Error: {e}")
        try:
            driver.quit()
        except:
            pass
        driver = create_driver()

    if (i + 1) % 50 == 0:
        df.to_csv("goodreads_1000_full.csv", index=False)
        print(f"  ✓ Progress saved at {i+1} books")

driver.quit()
df["source"] = "GoodReads"
df.to_csv("goodreads_1000_full.csv", index=False)
print("Done! Saved to goodreads_1000_full.csv")

[1/1000] ✓ Already scraped: The Hunger Games (The Hunger Games, #1)
[2/1000] ✓ Already scraped: Pride and Prejudice
[3/1000] ✓ Already scraped: To Kill a Mockingbird
[4/1000] ✓ Already scraped: Harry Potter and the Order of the Phoenix (Harry Potter, #5)
[5/1000] ✓ Already scraped: The Book Thief
[6/1000] ✓ Already scraped: Twilight (Twilight Saga, #1)
[7/1000] ✓ Already scraped: Animal Farm
[8/1000] ✓ Already scraped: J.R.R. Tolkien 4-Book Boxed Set: The Hobbit and The Lord of the Rings
[9/1000] ✓ Already scraped: The Chronicles of Narnia (The Chronicles of Narnia, #1-7)
[10/1000] ✓ Already scraped: The Fault in Our Stars
[11/1000] ✓ Already scraped: The Picture of Dorian Gray
[12/1000] ✓ Already scraped: The Lightning Thief (Percy Jackson and the Olympians, #1)
[13/1000] ✓ Already scraped: Wuthering Heights
[14/1000] ✓ Already scraped: The Giving Tree
[15/1000] ✓ Already scraped: Harry Potter and the Order of the Phoenix (Harry Potter, #5)
[16/1000] ✓ Already scraped: The Perks of Be

In [5]:
df_api = pd.read_csv("goodreads_1000_full.csv")
print(len(df_api))

1000


In [7]:
# Find duplicates
duplicates = df_goodreads[df_goodreads["title"].str.lower().str.strip().duplicated(keep=False)]
print("=== DUPLICATES ===")
print(duplicates[["rank", "title", "author"]])

# Find Arabic books
arabic = df_goodreads[df_goodreads["description"].str.contains(r'[\u0600-\u06FF]', na=False, regex=True)]
print("\n=== ARABIC BOOKS ===")
print(arabic[["rank", "title", "author"]])

=== DUPLICATES ===
     rank                                              title          author
2       3                              To Kill a Mockingbird      Harper Lee
3       4  Harry Potter and the Order of the Phoenix (Har...    J.K. Rowling
14     15  Harry Potter and the Order of the Phoenix (Har...    J.K. Rowling
33     34                                               1984   George Orwell
36     37                                     Fahrenheit 451    Ray Bradbury
82     83  The Fellowship of the Ring (The Lord of the Ri...  J.R.R. Tolkien
88     89                              To Kill a Mockingbird      Harper Lee
112   113                                     Fahrenheit 451    Ray Bradbury
131   132                                               1984   George Orwell
194   195  A Court of Mist and Fury (A Court of Thorns an...   Sarah J. Maas
324   325  The Fellowship of the Ring (The Lord of the Ri...  J.R.R. Tolkien
364   365  A Court of Thorns and Roses (A Court of Thorns

In [8]:
#replacing the non-english and droping duplicate books
# Step 1: Remove duplicate rows — keep first occurrence
df_goodreads = df_goodreads.drop_duplicates(subset="title", keep="first")
print(f"After removing duplicates: {len(df_goodreads)} books")

# Step 2: Drop the 4 Arabic books
arabic_indices = df_goodreads[df_goodreads["description"].str.contains(
    r'[\u0600-\u06FF\u0590-\u05FF]', na=False, regex=True
) & df_goodreads["title"].str.contains(
    r'[\u0600-\u06FF\u0590-\u05FF]', na=False, regex=True
)].index.tolist()

print(f"Arabic books to remove: {len(arabic_indices)}")
print(df_goodreads.loc[arabic_indices, "title"].tolist())

df_goodreads = df_goodreads.drop(arabic_indices)
print(f"After removing Arabic books: {len(df_goodreads)} books")

After removing duplicates: 991 books
Arabic books to remove: 4
['القرآن الكريم', 'زندگی مه آلود پریا', 'انتحار فاشل', 'العصبي']
After removing Arabic books: 987 books


In [ ]:
#fetching new books

# Open browser and scrape page 11 onwards for replacement books
options = webdriver.ChromeOptions()
options.add_argument("--disable-blink-features=AutomationEnabled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

replacement_books = []
page = 11

while len(replacement_books) < 13:
    url = f"https://www.goodreads.com/list/show/1.Best_Books_Ever?page={page}"
    print(f"Fetching page {page}...")
    
    driver.get(url)
    time.sleep(15)
    
    soup = BeautifulSoup(driver.page_source, "lxml")
    table = soup.find("table", {"class": "tableList"})
    
    if not table:
        print("No table found")
        break
    
    rows = table.find_all("tr")
    for row in rows:
        if len(replacement_books) >= 13:
            break
            
        title_tag  = row.find("a", class_="bookTitle")
        author_tag = row.find("a", class_="authorName")
        rating_tag = row.find("span", class_="minirating")
        rank_tag   = row.find("td", class_="number")
        img_tag    = row.find("img")

        title = title_tag.get_text(strip=True) if title_tag else None
        if not title:
            continue
            
        # Skip if already in dataset
        if title.lower().strip() in set(df_goodreads["title"].str.lower().str.strip()):
            continue

        avg_rating, num_ratings = None, None
        if rating_tag:
            rating_text = rating_tag.get_text(strip=True)
            try:
                avg_rating = float(rating_text.split()[0])
            except:
                pass
            try:
                num_ratings = int(rating_text.split("—")[1].strip().split()[0].replace(",", ""))
            except:
                pass

        replacement_books.append({
            "rank":          rank_tag.get_text(strip=True) if rank_tag else None,
            "title":         title,
            "author":        author_tag.get_text(strip=True) if author_tag else None,
            "avg_rating":    avg_rating,
            "num_ratings":   num_ratings,
            "cover_img_url": img_tag["src"] if img_tag else None,
            "goodreads_url": "https://www.goodreads.com" + title_tag["href"] if title_tag else None,
            "source":        "GoodReads"
        })
        print(f"  ✓ Added: {title}")

    page += 1
    time.sleep(random.uniform(3, 6))

driver.quit()
print(f"\nFound {len(replacement_books)} replacement books")

Fetching page 11...
  ✓ Added: Life After Life (Todd Family, #1)
  ✓ Added: Sea of Tranquility
  ✓ Added: Howards End
  ✓ Added: The Trumpet of the Swan
  ✓ Added: The Forgotten Garden
  ✓ Added: Kane & Abel (Kane & Abel, #1)
  ✓ Added: Wild Swans: Three Daughters of China
  ✓ Added: The Hunt for Red October (Jack Ryan, #3)
  ✓ Added: The White Tiger
  ✓ Added: Living Fearless: Exchanging the Lies of the World for the Liberating Truth of God
  ✓ Added: Tender Is the Night
  ✓ Added: The Bridges of Madison County
  ✓ Added: The Strength of the Few (Hierarchy, #2)

Found 13 replacement books


In [12]:
print(len(replacement_books))

13


In [ ]:
#Enriching the new books with the additional features
def scrape_book_details(url, driver):
    driver.get(url)
    time.sleep(random.uniform(8, 12))
    soup = BeautifulSoup(driver.page_source, "lxml")
    
    year_tag = soup.find("p", {"data-testid": "publicationInfo"})
    year = year_tag.get_text(strip=True) if year_tag else None

    genres_div = soup.find(class_="BookPageMetadataSection__genres")
    genre_links = genres_div.find_all("a") if genres_div else []
    genres = ", ".join([g.get_text(strip=True) for g in genre_links])

    desc = None
    desc_div = soup.find("div", {"data-testid": "description"})
    if desc_div:
        desc = desc_div.get_text(strip=True)
    else:
        for tag in soup.find_all(["span", "div"]):
            text = tag.get_text(strip=True)
            if len(text) > 100 and len(text) < 2000 and "{" not in text:
                desc = text
                break

    pages_tag = soup.find("p", {"data-testid": "pagesFormat"})
    pages = pages_tag.get_text(strip=True) if pages_tag else None

    series_tag = soup.find("h3", {"aria-label": lambda x: x and "series" in x.lower()})
    series = series_tag.get_text(strip=True) if series_tag else None

    return {"year_published": year, "genres": genres, "description": desc, "pages": pages, "series": series}


# Open driver
options = webdriver.ChromeOptions()
options.add_argument("--disable-blink-features=AutomationEnabled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Enrich and save
for book in replacement_books:
    print(f"Enriching: {book['title']}")
    details = scrape_book_details(book["goodreads_url"], driver)
    book.update(details)
    time.sleep(random.uniform(3, 6))

driver.quit()

df_goodreads = pd.read_csv("goodreads_1000_full.csv")
df_replacements = pd.DataFrame(replacement_books)
df_goodreads = pd.concat([df_goodreads, df_replacements], ignore_index=True)
print(f"Final GoodReads dataset: {len(df_goodreads)} books")
df_goodreads.to_csv("goodreads_1000_full.csv", index=False)
print("Saved!")

Enriching: Life After Life (Todd Family, #1)
Enriching: Sea of Tranquility
Enriching: Howards End
Enriching: The Trumpet of the Swan
Enriching: The Forgotten Garden
Enriching: Kane & Abel (Kane & Abel, #1)
Enriching: Wild Swans: Three Daughters of China
Enriching: The Hunt for Red October (Jack Ryan, #3)
Enriching: The White Tiger
Enriching: Living Fearless: Exchanging the Lies of the World for the Liberating Truth of God
Enriching: Tender Is the Night
Enriching: The Bridges of Madison County
Enriching: The Strength of the Few (Hierarchy, #2)
Final GoodReads dataset: 1013 books
Saved!


In [16]:
# Check for duplicates
print(f"Before dedup: {len(df_goodreads)}")
df_goodreads = df_goodreads.drop_duplicates(subset="title", keep="first").reset_index(drop=True)
print(f"After dedup: {len(df_goodreads)}")

df_goodreads.to_csv("goodreads_1000_full.csv", index=False)
print("Saved!")

Before dedup: 1013
After dedup: 1004
Saved!


In [ ]:
#overlapping books in both datasets
goodreads_titles = set(df_goodreads["title"].str.lower().str.strip())
df_api = pd.read_csv("openlibrary_1000.csv")
api_titles = set(df_api["title"].str.lower().str.strip())

overlap = goodreads_titles.intersection(api_titles)
print(f"Overlapping books between datasets: {len(overlap)}")

Overlapping books between datasets: 1


In [19]:
actually_foreign = ['القرآن الكريم', 'زندگی مه آلود پریا', 'انتحار فاشل', 'العصبي']
foreign_idx = df_goodreads[df_goodreads["title"].isin(actually_foreign)].index.tolist()
print(foreign_idx)

[476, 500, 634, 642]


Replacing 4 non-English books with English ones using the same function as before

In [20]:
existing_titles = set(df_goodreads["title"].str.lower().str.strip()) | set(df_api["title"].str.lower().str.strip())

options = webdriver.ChromeOptions()
options.add_argument("--disable-blink-features=AutomationEnabled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

replacement_books = []
page = 12

while len(replacement_books) < 4:
    url = f"https://www.goodreads.com/list/show/1.Best_Books_Ever?page={page}"
    print(f"Fetching page {page}...")
    driver.get(url)
    time.sleep(15)
    soup = BeautifulSoup(driver.page_source, "lxml")
    table = soup.find("table", {"class": "tableList"})
    if not table:
        break
    rows = table.find_all("tr")
    for row in rows:
        if len(replacement_books) >= 4:
            break
        title_tag  = row.find("a", class_="bookTitle")
        author_tag = row.find("a", class_="authorName")
        rating_tag = row.find("span", class_="minirating")
        rank_tag   = row.find("td", class_="number")
        img_tag    = row.find("img")
        title = title_tag.get_text(strip=True) if title_tag else None
        if not title or title.lower().strip() in existing_titles:
            continue
        if not title.isascii():
            continue
        avg_rating, num_ratings = None, None
        if rating_tag:
            rating_text = rating_tag.get_text(strip=True)
            try:
                avg_rating = float(rating_text.split()[0])
            except:
                pass
            try:
                num_ratings = int(rating_text.split("—")[1].strip().split()[0].replace(",", ""))
            except:
                pass
        replacement_books.append({
            "rank":          rank_tag.get_text(strip=True) if rank_tag else None,
            "title":         title,
            "author":        author_tag.get_text(strip=True) if author_tag else None,
            "avg_rating":    avg_rating,
            "num_ratings":   num_ratings,
            "cover_img_url": img_tag["src"] if img_tag else None,
            "goodreads_url": "https://www.goodreads.com" + title_tag["href"] if title_tag else None,
            "source":        "GoodReads"
        })
        print(f"  ✓ Added: {title}")
    page += 1
    time.sleep(random.uniform(3, 6))

print(f"\nFound {len(replacement_books)} replacement books")

# Enrich with detail fields
for book in replacement_books:
    print(f"Enriching: {book['title']}")
    details = scrape_book_details(book["goodreads_url"], driver)
    book.update(details)
    time.sleep(random.uniform(3, 6))

driver.quit()

# Replace the 4 rows
for idx, book in zip([476, 500, 634, 642], replacement_books):
    for col, val in book.items():
        if col in df_goodreads.columns:
            df_goodreads.at[idx, col] = val

df_goodreads.to_csv("goodreads_1000_full.csv", index=False)
print("Saved!")

# Final check
remaining = df_goodreads[df_goodreads["title"].str.contains(r'[^\x00-\x7F]', na=False, regex=True)]
actually_foreign_remaining = [t for t in remaining["title"].tolist() if not any(c.isascii() for c in t.replace("'","").replace("é","e").replace("ó","o"))]
print(f"Arabic/Persian titles remaining: {len([t for t in remaining['title'].tolist() if not t.isascii() or any(ord(c) > 1000 for c in t)])}")

Fetching page 12...
  ✓ Added: A Million Little Pieces
  ✓ Added: We
  ✓ Added: Gabriel's Inferno (Gabriel's Inferno, #1)
  ✓ Added: Light Bringer (Red Rising Saga, #6)

Found 4 replacement books
Enriching: A Million Little Pieces
Enriching: We
Enriching: Gabriel's Inferno (Gabriel's Inferno, #1)
Enriching: Light Bringer (Red Rising Saga, #6)
Saved!
Arabic/Persian titles remaining: 42


In [21]:
#check
print(f"GoodReads shape: {df_goodreads.shape}")
print(f"Open Library shape: {df_api.shape}")

# Overlap check
gr_titles = set(df_goodreads["title"].str.lower().str.strip())
api_titles = set(df_api["title"].str.lower().str.strip())
overlap = gr_titles.intersection(api_titles)
print(f"Overlap between datasets: {len(overlap)}")

# Duplicates
print(f"GoodReads duplicates: {df_goodreads['title'].duplicated().sum()}")
print(f"Open Library duplicates: {df_api['title'].duplicated().sum()}")

GoodReads shape: (1004, 13)
Open Library shape: (1000, 12)
Overlap between datasets: 1
GoodReads duplicates: 0
Open Library duplicates: 0


In [22]:
print("GoodReads columns:", df_goodreads.columns.tolist())
print("Open Library columns:", df_api.columns.tolist())

# What's in GoodReads but not Open Library
missing = set(df_goodreads.columns) - set(df_api.columns)
print("Missing from Open Library:", missing)

GoodReads columns: ['rank', 'title', 'author', 'avg_rating', 'num_ratings', 'cover_img_url', 'goodreads_url', 'source', 'year_published', 'genres', 'description', 'pages', 'series']
Open Library columns: ['title', 'author', 'year_published', 'isbn', 'publisher', 'language', 'genres', 'pages', 'cover_img_url', 'ol_key', 'source', 'description']
Missing from Open Library: {'goodreads_url', 'avg_rating', 'rank', 'num_ratings', 'series'}


In [23]:
df_goodreads = pd.read_csv("goodreads_1000_full.csv")

arabic = df_goodreads[df_goodreads["title"].str.contains(
    r'[\u0600-\u06FF\u0590-\u05FF]', na=False, regex=True
)]
print(f"Arabic/Persian titles: {len(arabic)}")
print(arabic[["title"]].to_string())

Arabic/Persian titles: 4
                  title
476       القرآن الكريم
500  زندگی مه آلود پریا
634         انتحار فاشل
642              العصبي


In [24]:
existing_titles = set(df_goodreads["title"].str.lower().str.strip()) | set(df_api["title"].str.lower().str.strip())

options = webdriver.ChromeOptions()
options.add_argument("--disable-blink-features=AutomationEnabled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

replacement_books = []
page = 13

while len(replacement_books) < 4:
    url = f"https://www.goodreads.com/list/show/1.Best_Books_Ever?page={page}"
    print(f"Fetching page {page}...")
    driver.get(url)
    time.sleep(15)
    soup = BeautifulSoup(driver.page_source, "lxml")
    table = soup.find("table", {"class": "tableList"})
    if not table:
        break
    rows = table.find_all("tr")
    for row in rows:
        if len(replacement_books) >= 4:
            break
        title_tag  = row.find("a", class_="bookTitle")
        author_tag = row.find("a", class_="authorName")
        rating_tag = row.find("span", class_="minirating")
        rank_tag   = row.find("td", class_="number")
        img_tag    = row.find("img")
        title = title_tag.get_text(strip=True) if title_tag else None
        if not title or not title.isascii():
            continue
        if title.lower().strip() in existing_titles:
            continue
        avg_rating, num_ratings = None, None
        if rating_tag:
            rating_text = rating_tag.get_text(strip=True)
            try:
                avg_rating = float(rating_text.split()[0])
            except:
                pass
            try:
                num_ratings = int(rating_text.split("—")[1].strip().split()[0].replace(",", ""))
            except:
                pass
        replacement_books.append({
            "rank":          rank_tag.get_text(strip=True) if rank_tag else None,
            "title":         title,
            "author":        author_tag.get_text(strip=True) if author_tag else None,
            "avg_rating":    avg_rating,
            "num_ratings":   num_ratings,
            "cover_img_url": img_tag["src"] if img_tag else None,
            "goodreads_url": "https://www.goodreads.com" + title_tag["href"] if title_tag else None,
            "source":        "GoodReads"
        })
        print(f"  ✓ Added: {title}")
    page += 1
    time.sleep(random.uniform(3, 6))

# Enrich with detail fields
for book in replacement_books:
    print(f"Enriching: {book['title']}")
    details = scrape_book_details(book["goodreads_url"], driver)
    book.update(details)
    time.sleep(random.uniform(3, 6))

driver.quit()

# Replace the 4 rows and save immediately
for idx, book in zip([476, 500, 634, 642], replacement_books):
    for col, val in book.items():
        if col in df_goodreads.columns:
            df_goodreads.at[idx, col] = val
    df_goodreads.to_csv("goodreads_1000_full.csv", index=False)
    print(f"  ✓ Saved row {idx}: {book['title']}")

# Final verification
arabic_check = df_goodreads[df_goodreads["title"].str.contains(r'[\u0600-\u06FF\u0590-\u05FF]', na=False, regex=True)]
print(f"\nArabic titles remaining: {len(arabic_check)}")

Fetching page 13...
  ✓ Added: Fantasy Lover (Hunter Legends, #1)
  ✓ Added: Fenway and Hattie (Fenway and Hattie, #1)
  ✓ Added: Silas Marner
  ✓ Added: Heartstopper: Volume Two (Heartstopper, #2)
Enriching: Fantasy Lover (Hunter Legends, #1)
Enriching: Fenway and Hattie (Fenway and Hattie, #1)
Enriching: Silas Marner
Enriching: Heartstopper: Volume Two (Heartstopper, #2)
  ✓ Saved row 476: Fantasy Lover (Hunter Legends, #1)
  ✓ Saved row 500: Fenway and Hattie (Fenway and Hattie, #1)
  ✓ Saved row 634: Silas Marner
  ✓ Saved row 642: Heartstopper: Volume Two (Heartstopper, #2)

Arabic titles remaining: 0


/var/folders/xg/zchszbrn7h32p8rl8v_7dg4w0000gn/T/ipykernel_28303/61942515.py:73: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1201' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_goodreads.at[idx, col] = val


In [26]:
#check
df_goodreads = pd.read_csv("goodreads_1000_full.csv")
df_api = pd.read_csv("openlibrary_1000.csv")

print(f"GoodReads: {df_goodreads.shape}")
print(f"Open Library: {df_api.shape}")

overlap = set(df_goodreads["title"].str.lower().str.strip()) & set(df_api["title"].str.lower().str.strip())
print(f"Overlap: {len(overlap)}")

arabic_gr = df_goodreads[df_goodreads["title"].str.contains(r'[\u0600-\u06FF\u0590-\u05FF]', na=False, regex=True)]
arabic_ol = df_api[df_api["title"].str.contains(r'[\u0600-\u06FF\u0590-\u05FF]', na=False, regex=True)]
print(f"Arabic in GoodReads: {len(arabic_gr)}")
print(f"Arabic in Open Library: {len(arabic_ol)}")

print(f"GoodReads duplicates: {df_goodreads['title'].duplicated().sum()}")
print(f"Open Library duplicates: {df_api['title'].duplicated().sum()}")

GoodReads: (1004, 13)
Open Library: (1000, 15)
Overlap: 0
Arabic in GoodReads: 0
Arabic in Open Library: 0
GoodReads duplicates: 0
Open Library duplicates: 0
